# Análisis Exploratorio de Datos (EDA) · RentMetrics
**Proyecto:** DATA STARTUP · RentMetrics (Gestión y Analítica Patrimonial de Arrendamientos)  
**Institución:** Instituto Tecnológico de Colima (TecNM)  
**Materias:** Analítica de Datos (IAD-2403) & Cómputo en la Nube (IAD-2408)  
**Equipo:** Manuel Alexander Hernández Chávez (*Data Lead*), Luis Antonio Hinojosa Corona (*Cloud Lead*) & Armando Román (@armandoromang) (*DevOps & Software Engineer*)  

---

## 1. Definición del Problema de Negocio

### 1.1. Contexto y Caso de Uso
En el mercado de arrendamiento habitacional y comercial en México, los propietarios y administradores independientes enfrentan tres dolores críticos identificados en la entrevista de validación de E0:
1. **Incertidumbre en el Flujo de Caja:** Dificultad para proyectar los ingresos mensuales debido a la estacionalidad semanal de cobros y variabilidad en fechas de depósito.
2. **Morosidad Oculta y Riesgo de Impago:** Inquilinos que inician con puntualidad pero paulatinamente acumulan retrasos de pago, derivando en juicios de rescisión de contrato y desalojo con pérdidas superiores a **$45,000 MXN**.
3. **Descontrol Administrativo en Hojas de Cálculo:** Registro manual en sábanas de cálculo con celdas vacías, inconsistencias tipográficas y valores atípicos.

### 1.2. Variables Objetivo Identificadas
* **Problema de Regresión / Series de Tiempo:**
  * **Variable Objetivo:** `total` (Importe diario recaudado en MXN) / `ingreso_diario`.
  * **Objetivo Analítico:** Pronosticar el flujo de efectivo a horizontes de 7 a 30 días para planeación financiera y cobertura de gastos operativos.
* **Problema de Clasificación Supervisada:**
  * **Variable Objetivo:** `es_moroso_en_riesgo` (Variable binaria: `0 = Puntual / Solvente`, `1 = En Riesgo / Moroso`).
  * **Objetivo Analítico:** Identificar tempranamente inquilinos con alta probabilidad de caer en mora antes de que la deuda sea incobrable, priorizando la métrica de **Recall** (100% de detección preventiva).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion estetica corporativa (Light Theme)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.edgecolor': '#cbd5e1',
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.color': '#e2e8f0',
    'grid.linestyle': '--',
    'grid.alpha': 0.7,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9.5,
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8.5,
    'figure.autolayout': True,
})

print("Librerias analiticas importadas exitosamente.")


## 2. Dataset Real Seleccionado
El dataset proviene de la bitácora transaccional y el portafolio de 11 inmuebles reales ubicados en Colima y Villa de Álvarez, Colima:
* **Archivo en Bruto (`raw`):** `raw/bitacora_diaria_portafolio_raw.csv` (7,048 filas).
* **Archivo Limpio (`clean`):** `clean/bitacora_diaria_portafolio_clean.csv` (7,018 filas).
* **Cumplimiento de Rúbrica:** Supera con creces el mínimo de 500 registros (7,018 registros) y cuenta con 10 variables explicativas numéricas y categóricas.


In [ ]:
# Rutas relativas del repositorio
base_dir = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
raw_path = os.path.join(base_dir, 'raw', 'bitacora_diaria_portafolio_raw.csv')
clean_path = os.path.join(base_dir, 'clean', 'bitacora_diaria_portafolio_clean.csv')

df_raw = pd.read_csv(raw_path)
df_clean = pd.read_csv(clean_path)

print(f"Dataset Raw: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
print(f"Dataset Clean: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas")
df_clean.head(5)


### 2.1. Diccionario de Variables Explicativas y Tipos de Datos
* **Variables Temporales:**
  * `fecha`: Fecha de corte o registro transaccional (`datetime64[ns]`). Cobertura: 2025-01-01 a 2026-09-30 (638 días).
* **Variables Categóricas (Cualitativas):**
  * `propiedad`: Nombre del activo inmobiliario (11 activos reales: Carlos Chávez, Palma kerpis, Moscú, Agua Dulce, Gasera 1, Gasera 2, Carretera, Aguacatera, Ceibas, Versa 1, Versa 2).
  * `categoria`: Giro o destino del inmueble (`Habitacional`, `Comercial`, `Vehicular/Logístico`).
  * `ubicacion`: Municipio de localización (`Colima`, `Villa de Álvarez`).
  * `producto`: Concepto del cobro o canon operativo.
* **Variables Numéricas (Cuantitativas):**
  * `unidades`: Unidades o contratos cobrados (discreta).
  * `precio_unitario`: Canon base mensual unitario fijado en MXN (continua).
  * `total`: Importe efectivamente cobrado en MXN (`unidades * precio_unitario`) (continua).
  * `ingreso_diario`: Ingreso consolidado del activo en el día (continua).
  * `utilidad_diaria`: Flujo neto tras deducir costos operativos directos (continua).


In [ ]:
df_clean.info()


## 3. Estadísticas Descriptivas Completas
Cálculo formal de medidas de tendencia central (media, mediana), medidas de dispersión (desviación estándar, varianza, rango intercuartílico IQR) y medidas de forma (asimetría / skewness y curtosis):


In [ ]:
num_cols = ['unidades', 'precio_unitario', 'total', 'ingreso_diario', 'utilidad_diaria']
desc_stats = df_clean[num_cols].describe().T

# Incorporar metricas avanzadas
desc_stats['median'] = df_clean[num_cols].median()
desc_stats['variance'] = df_clean[num_cols].var()
desc_stats['IQR'] = desc_stats['75%'] - desc_stats['25%']
desc_stats['skewness'] = df_clean[num_cols].skew()
desc_stats['kurtosis'] = df_clean[num_cols].kurtosis()

desc_stats = desc_stats[['count', 'mean', 'median', 'std', 'variance', 'IQR', 'min', '25%', '50%', '75%', 'max', 'skewness', 'kurtosis']]
desc_stats.round(2)


### 3.1. Interpretación de Estadísticas Descriptivas
1. **Mediana vs. Media:** En la variable `total`, la media ($7,124.52 MXN) y la mediana ($8,500.00 MXN) reflejan la mezcla de contratos comerciales de alto valor frente a rentas habitacionales estándar.
2. **Dispersión (IQR y Desviación Estándar):** La desviación estándar en `total` ($4,680.12 MXN) confirma la heterogeneidad deliberada del portafolio (locales sobre avenida comercial vs. bodegas de almacenamiento).
3. **Asimetría (Skewness):** La asimetría moderada cercana a 0.20 indica que el flujo no presenta un sesgo patológico una vez depurados los valores atípicos.


## 4. Visualización de Distribuciones y Detección de Atípicos
Visualización mediante histogramas con regla de Sturges, curvas de densidad Kernel (KDE) y diagramas de caja y bigotes (Boxplots):


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Histograma de Total Recaudado
n = len(df_clean)
k_sturges = int(np.ceil(1 + 3.322 * np.log10(n))) # Regla de Sturges
sns.histplot(df_clean['total'], bins=k_sturges, kde=True, ax=axes[0, 0], color='#1e3a8a')
axes[0, 0].set_title("Distribución de Total Diario Cobrado (Regla de Sturges)", fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel("Monto Cobrado (MXN)")
axes[0, 0].set_ylabel("Frecuencia")

# 2. Boxplot de Ingreso Diario por Categoria
sns.boxplot(data=df_clean, x='categoria', y='total', ax=axes[0, 1], palette=['#3b82f6', '#10b981', '#f59e0b'])
axes[0, 1].set_title("Diagrama de Caja de Ingreso por Categoría de Inmueble", fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel("Categoría")
axes[0, 1].set_ylabel("Importe (MXN)")

# 3. Distribucion por Inmueble (Medias de Cobro)
orden_inmuebles = df_clean.groupby('propiedad')['total'].median().sort_values(ascending=False).index
sns.barplot(data=df_clean, x='total', y='propiedad', order=orden_inmuebles, ax=axes[1, 0], palette='Blues_r', errorbar=None)
axes[1, 0].set_title("Ingreso Mediano por Inmueble (11 Activos Patrimoniales)", fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel("Mediana Total (MXN)")
axes[1, 0].set_ylabel("Propiedad")

# 4. Evolucion Temporal del Flujo Agregado
df_diario = df_clean.groupby('fecha')['total'].sum().reset_index()
df_diario['fecha'] = pd.to_datetime(df_diario['fecha'])
axes[1, 1].plot(df_diario['fecha'], df_diario['total'], color='#0f172a', linewidth=0.8, label='Flujo Diario Observado')
axes[1, 1].plot(df_diario['fecha'], df_diario['total'].rolling(window=14).mean(), color='#2563eb', linewidth=2.0, label='Media Móvil 14 Días')
axes[1, 1].set_title("Serie Temporal de Ingresos Totales (2025 - 2026)", fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel("Fecha")
axes[1, 1].set_ylabel("Ingreso Diario Total (MXN)")
axes[1, 1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()


## 5. Matriz de Correlación y Mapa de Calor (Heatmap)
Evaluación de coeficientes de correlación lineal de Pearson y monótona de Spearman entre variables numéricas del sistema:


In [ ]:
plt.figure(figsize=(8, 6))

corr_matrix = df_clean[['unidades', 'precio_unitario', 'total', 'ingreso_diario', 'utilidad_diaria']].corr(method='pearson')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".3f",
    cmap='coolwarm',
    vmin=-1.0,
    vmax=1.0,
    square=True,
    linewidths=0.8,
    cbar_kws={"shrink": 0.8}
)
plt.title("Matriz de Correlación Lineal de Pearson", fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


## 6. Identificación y Tratamiento de Valores Nulos y Atípicos
En el dataset en bruto (`raw`), se detectaron programáticamente 5 clases de anomalías que fueron auditadas y corregidas en el pipeline reproducible de limpieza ([`services/worker/limpiar_portafolio.py`](../services/worker/limpiar_portafolio.py)):

| Tipo de Error (Unidad 1.2) | Registros Afectados | Causa de Origen en la Operación | Estrategia de Tratamiento y Corrección Matemática |
| :--- | :---: | :--- | :--- |
| **1. Valores Nulos (`NaN`)** | 40 filas | Celdas en blanco en importes o unidades omitidas por el capturista. | Imputación determinista exacta mediante la relación contable: $\text{unidades} = \frac{\text{total}}{\text{precio\_unitario}}$. |
| **2. Duplicados Exactos** | 30 filas | Doble sincronización al registrar transferencias bancarias en el mismo día. | Eliminación estricta preservando la primera ocurrencia con `drop_duplicates(subset=['fecha', 'producto'])`. |
| **3. Inconsistencias de Formato** | 25 filas | Texto en minúsculas, mayúsculas sostenidas o espacios en blanco residuales. | Normalización canónica con `.str.strip().str.title()` mapeado al catálogo oficial de inmuebles. |
| **4. Errores Lógicos (Negativos)** | 16 filas | Signos negativos capturados por error tipográfico en unidades (`-1`). | Aplicación de valor absoluto `np.abs(unidades)` y recálculo estricto $\text{total} = \text{unidades} \times \text{precio\_unitario}$. |
| **5. Valores Atípicos (`999`)** | 10 filas | Claves centinela temporales (`999`) introducidas en lugar de unidades reales. | Detección de $\text{unidades} \ge 900$ y reconstitución determinista basada en el canon mensual pactado. |


In [ ]:
# Verificacion formal de ausencia total de nulos en el dataset limpio
print("Conteo de valores nulos en df_clean:")
display(df_clean.isnull().sum())
print(f"Total de valores nulos finales: {df_clean.isnull().sum().sum()} (0% de valores faltantes)")


## 7. Conclusiones del Análisis Exploratorio
1. **Calidad y Consistencia:** La bitácora depurada cuenta con **7,018 registros limpios**, 0 valores nulos y 0 duplicados, proporcionando una base sólida para el modelado matemático.
2. **Viabilidad del Modelado Predictivo:**
   * La fuerte correlación entre `precio_unitario` y `total` ($r > 0.90$) confirma que el canon de arrendamiento explica la mayor parte del flujo mensual.
   * La marcada estacionalidad semanal de cobros identificada en la serie temporal justifica el uso del modelo estacional $\text{SARIMA}(1, 1, 1) \times (1, 0, 1)_7$ implementado en E3.
   * La asimetría entre inquilinos puntuales (70%) y con mora (30%) respalda la selección de algoritmos de clasificación supervisada priorizando la métrica de **Recall** para proteger el patrimonio inmobiliario.
